In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
INPUT_FOLDER = './' # ./kaggle/input
import os
fnames = []
for dirname, _, filenames in os.walk(INPUT_FOLDER):
    for filename in filenames:
        fnames.append(filename)
        print(os.path.join(dirname, filename))
        break
        ''

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

! pip install librosa

In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import librosa
import librosa.display
import matplotlib.pyplot as plt
import random
import torch

import warnings
warnings.filterwarnings("ignore")

In [4]:
genres = ["blues", "classical", "country", "disco", "hiphop",
          "jazz", "metal", "pop", "reggae", "rock"]

In [5]:
from sklearn.model_selection import train_test_split

In [9]:
# Milestone 1
#Q1
recipes = [{"genre": g, "i": i} for g in genres for i in range(100)]

train_recipes, val_recipes = train_test_split(
    recipes, test_size=0.2, shuffle=True, random_state=42
)
print(len(val_recipes))

200


In [33]:
#Q2
sample_dir = r'C:\Users\rramk\Downloads\dl-genai-project-26-t1\messy_mashup\genres_stems\rock\rock.00097'

t_length = 160000
def fix_length(y, target=t_length):
    if len(y) < target:
        return np.pad(y, (0, target - len(y)))
    return y[:target]

vocals = librosa.load(sample_dir+'/vocals.wav',sr=16000, duration=10)[0]
drums = librosa.load(sample_dir+'/drums.wav',sr=16000, duration=10)[0]
other = librosa.load(sample_dir+'/other.wav',sr=16000, duration=10)[0]
bass = librosa.load(sample_dir+'/bass.wav',sr=16000, duration=10)[0]

noise = fix_length(librosa.load(r'C:\Users\rramk\Downloads\dl-genai-project-26-t1\messy_mashup\ESC-50-master\audio\5-251971-A-47.wav',sr=16000, duration=10)[0])

result = (vocals) + (drums) + (bass) + (other) + (noise*.2)
result.shape

(160000,)

!pip install transformers

In [ ]:
#Q3

In [48]:
from transformers import AutoFeatureExtractor, ASTForAudioClassification

In [35]:
extractor = AutoFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [40]:
inp = np.ones(160000)
inp_f = extractor(inp, sampling_rate=16000, return_tensors='pt')

In [44]:
inp_f['input_values'].squeeze(0).shape

torch.Size([1024, 128])

In [49]:
#Q4
extractor = ASTForAudioClassification.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593", num_labels=10, ignore_mismatched_sizes=True)

config.json: 0.00B [00:00, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [50]:
sum(p.numel() for p in extractor.parameters() if p.requires_grad) 

86196490

In [51]:
# Q5
y = np.array([-0.85, 0.40, 0.20, -0.10])
y = y / (np.max(np.abs(y)) + 1e-9)

In [52]:
y

array([-1.        ,  0.47058823,  0.23529412, -0.11764706])